# Learning module 6 — Install and select the Appium server package

> **Appium** is the execution layer for our UI tests. Our agent decides what action to take; Appium provides the connection and commands needed to perform it on the device.

For example, our agent can request:
- Inspect the current UI elements.
- Find a particular control.
- Tap it or enter text.
- Read its displayed value.
- Capture a screenshot.
Appium manages the automation session and returns results or errors. Our agent and analyzer use those observations to determine whether the scenario succeeded. [Appium architecture](https://appium.io/docs/en/latest/intro/appium/)

**Small milestone:** install a pinned server package and verify its CLI, without starting its HTTP service or creating a device session.

Appium **3.9.0** is installed as an exact development-tool dependency in the server's private npm project. The installation used our local Node `24.22.2` and npm `11.19.1`. Python dependencies remain managed separately in the Python project.

| Artifact | Purpose |
| --- | --- |
| `server/package.json` | Records the exact Appium dependency and local tooling commands. |
| `server/package-lock.json` | Records the resolved npm dependency tree and integrity information. |
| `server/node_modules/` | Contains installed packages; excluded from Git. |
| `server/.tools/appium/` | Isolated Appium home for extension metadata and working state; excluded from Git. |
| `server/.tools/npm-cache/` | Local npm download/cache data; excluded from Git. |

The install command was `npm install --no-fund --registry=https://registry.npmjs.org`, run from `server` with the local Node first on PATH and project-local Appium/cache directories configured. Once the manifest and lockfile exist, `npm ci` can restore the locked package tree under the configured Node environment.

The compatibility choice is deliberate: Appium 3.9.0 declares support for our Node version. The intended later XCUITest 12.22.8 package declares an Appium 3 peer requirement and falls in the documented iOS 27-compatible range. That driver is **not installed in this step**. The current driver 13 line requires Appium 4, so server and driver majors must be selected together. [Appium requirements](https://appium.io/docs/en/latest/quickstart/requirements/), [XCUITest matrix](https://appium.github.io/appium-xcuitest-driver/latest/getting-started/system-requirements/)

Verified so far: the local CLI reports `3.9.0`, the manifest and lockfile agree, and the installed-driver list in the isolated Appium home is empty. Appium installation and server startup are distinct steps; the official guide also separates server setup from driver installation. [Appium installation guide](https://appium.io/docs/en/latest/quickstart/install/)

**Audit status — November 2, 2026:** npm reports nine moderate affected-package findings, originating from two underlying advisories: [glob cancellation](https://github.com/advisories/GHSA-3g3m-2fjr-6pw4) and [minimatch pattern amplification](https://github.com/advisories/GHSA-g3hv-75wg-m249). The report contains no high or critical findings. Its automated remediation proposes Appium 2.0.1, a major downgrade incompatible with our intended driver line; no forced downgrade or unverified dependency-major override was applied. These findings remain unresolved. Compatible remediation and affected runtime paths need review before server launch; successful installation is not security readiness.

**Run the Node setup cell in this notebook first.** The next cell selects the already installed Appium CLI and verifies its pinned version. It does not run `npm install`, start the server, or install a driver.

## This notebook

Part **06 — Select the pinned Appium package** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Kernel environment plus Appium version check.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.

### Setup for this notebook

These source-only prerequisite cells are repeated intentionally so the lesson is runnable in a fresh kernel. They configure local values/environment and inspect tool versions; they do not launch an Appium server, create a session, or tap/type in the app. The original lessons remain linked for explanation.


In [ ]:
# Setup: Local Node activation (lesson 05)
import os
from pathlib import Path
import shutil
import subprocess

notebook_cwd = Path.cwd().resolve()
server_candidates = (notebook_cwd, notebook_cwd / "server", *notebook_cwd.parents)
server_dir = next((candidate for candidate in server_candidates
                   if (candidate / ".node-version").is_file()
                   and (candidate / "package.json").is_file()
                   and (candidate / "sample_scenarios").is_dir()), None)
if server_dir is None:
    raise RuntimeError("Run from the project root, server, or sample_scenarios folder.")

node_version = (server_dir / ".node-version").read_text().strip()
node_bin = server_dir / ".tools" / f"node-v{node_version}-darwin-arm64" / "bin"
if not (node_bin / "node").is_file():
    raise FileNotFoundError(f"Project-local Node is missing: {node_bin}")

path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(node_bin)]
os.environ["PATH"] = os.pathsep.join([str(node_bin), *path_entries])

print("Selected Node:", shutil.which("node"))
print("Node version:", subprocess.check_output(["node", "--version"], text=True, timeout=10).strip())
print("npm version:", subprocess.check_output(["npm", "--version"], text=True, timeout=10).strip())


In [10]:
import json
import os
import shutil
import subprocess

if "server_dir" not in globals() or "node_bin" not in globals():
    raise RuntimeError("Run the Node setup cell in this notebook first.")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Reactivate the project-local Node runtime first.")

appium_bin_dir = server_dir / "node_modules" / ".bin"
appium_executable = appium_bin_dir / "appium"
if not appium_executable.is_file():
    raise FileNotFoundError("The project-local Appium package is missing.")

os.environ["APPIUM_HOME"] = str(server_dir / ".tools" / "appium")
os.environ["npm_config_cache"] = str(server_dir / ".tools" / "npm-cache")
path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(appium_bin_dir)]
os.environ["PATH"] = os.pathsep.join([str(appium_bin_dir), *path_entries])

expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
actual_version = subprocess.check_output(
    [str(appium_executable), "--version"], cwd=server_dir, text=True, timeout=30
).strip()
if actual_version != expected_version:
    raise RuntimeError(f"Expected Appium {expected_version}, found {actual_version}.")

print("Selected Appium:", shutil.which("appium"))
print("Appium version:", actual_version)
print("Appium home:", os.environ["APPIUM_HOME"])


Selected Appium: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/node_modules/.bin/appium
Appium version: 3.9.0
Appium home: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium


dbug Appium Creating hash file directory: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium/node_modules/.cache/appium


### Read the Appium selection, then pause

The selected executable should be inside `server/node_modules/.bin/`, and the version should be `3.9.0` for this locked installation.

`PATH` selects the executable. `APPIUM_HOME` selects Appium's extension/work directory; it does not identify Movies or the Simulator. A driver-list command may initialize metadata there even when no driver is installed. [Managing Appium extensions](https://appium.io/docs/en/latest/guides/managing-exts/)

The notebook now selects both local Node and local Appium for its kernel and child processes. Another terminal needs its own environment selection; it does not inherit changes made inside this kernel. The version check invokes the exact installed binary, with no `npx` download fallback.

A fresh [readiness notebook](04_environment_readiness.ipynb) will not inherit this kernel's PATH or Appium home. To inspect the activated environment there, run its Node/Appium activation in the same kernel first. The Python client, XCUITest driver, and HTTP server are separate remaining checks. Older stored outputs are snapshots until their cells are rerun.

**Pause here:** distinguish installed package → running server → driver-backed device session. We have verified the first stage. The next driver milestone is installing the pinned compatible XCUITest package and checking its prerequisites; the recorded audit findings also require follow-up before server launch.

### Concept recap — Appium, XCUITest, and a UI journey

**Appium is the execution layer for our UI tests.** Our executor agent decides what action to take. Appium exposes the automation connection and commands needed to carry out supported actions and collect observations.

Through the selected driver, those commands can inspect the current UI, find a control, tap or enter text, read displayed values, and capture screenshots. Appium manages automation sessions and returns results or errors. The agent and analyzer use the evidence to evaluate the scenario. [How Appium works](https://appium.io/docs/en/latest/intro/appium/)

**The Appium XCUITest driver is the iOS adapter.** Appium has a common command interface; operating an iOS app requires Apple's platform automation mechanisms. The driver connects those two layers.

| Component | Responsibility |
| --- | --- |
| Human | Describe the intended behavior and expected outcome. |
| Our executor agent | Interpret the request, inspect observations, and choose the next permitted action. |
| MCP adapter / Appium client | Expose controlled tools and turn their calls into automation requests. |
| Appium server | Accept requests, manage sessions, and route commands to the selected driver. |
| Appium XCUITest driver | Implement the supported iOS automation commands. |
| WebDriverAgent / Apple XCTest | Provide the underlying iOS inspection and interaction mechanisms. |
| Analyzer | Compare observed behavior with the scenario's assertions. |

Apple's UI-testing technology and Appium's XCUITest driver are distinct components. The driver uses WebDriverAgent, which relies on Apple's testing framework. [XCUITest driver overview](https://appium.github.io/appium-xcuitest-driver/latest/overview/)

Our proposed command path is:

```mermaid
flowchart TD
    Agent["Our executor agent"] --> Tools["MCP adapter / Appium client"]
    Tools --> Server["Appium server"]
    Server --> Driver["XCUITest driver"]
    Driver --> Apple["WebDriverAgent / XCTest"]
    Apple --> App["Movies on iOS Simulator"]
```

Results and observations return through the same layers. The diagram describes the intended system; the agent and adapter have not been implemented in this notebook.

For our Interstellar journey:

| Step | Agent/system decision | Automation work through Appium and the driver |
| --- | --- | --- |
| Reach search | Inspect the current screen and identify an appropriate control, if navigation is needed. | Return UI information, locate the observed target, and perform the permitted tap. |
| Enter the query | Select the observed search field and supply Interstellar. | Enter the text and return the command result. |
| Select the movie | Identify the 2014 movie using actual result evidence. | Locate and open the selected result. |
| Inspect details | Request evidence for the title and release year. | Return the supported displayed values and screenshots. |
| Evaluate | Analyzer compares the actual values with the expected title/year. | Supply the execution evidence; a successful click alone does not establish a pass. |

The executor repeats **observe → choose → execute → observe** to handle a deeper journey. Appium does not interpret the English business goal or supply a complete navigation graph. Our agent combines the scenario, approved app knowledge, and current observations to determine the route.

The app continues using its normal real backend integration. UI automation alone does not prove that a fresh backend call succeeded; any such claim requires its own supporting evidence.

At the end of module 6, we have verified the **Appium server package and CLI**. Installing the XCUITest driver adds the iOS automation capability. Starting the server process and creating a driver-backed device session are subsequent milestones. This recap adds documentation only.

[Index](README.md) · [Previous lesson](05_local_node_runtime.ipynb) · [Next lesson](07_xcuitest_driver.ipynb)